# RetinaMNIST Representations Experiment (PCA vs AE vs MAE)

This Colab notebook orchestrates a complete experiment on RetinaMNIST:
- **PCA** on flattened pixels
- **Convolutional Autoencoder (AE)** latent vectors
- **Pretrained ViT/MAE encoder** CLS-token features (via `timm`)

All three representations are evaluated with the **same MLP classifier head** (trained separately per representation).

All outputs (plots + saved figures) are written to `./outputs/`.

## SECTION 0 — Setup

In [ ]:
# Install dependencies (Colab)
!pip -q install medmnist timm umap-learn ipywidgets seaborn tqdm scikit-learn

import os
import sys
import time
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# (Option A) Clone project repo (uncomment and set your repo)
# !git clone <YOUR_REPO_URL>

# (Option B) Mount Google Drive (uncomment)
# from google.colab import drive
# drive.mount('/content/drive')

# Add project root to sys.path
PROJECT_ROOT = Path.cwd() / "project"
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Expected ./project folder next to this notebook. "
        "If using git clone or Drive mount, cd into the correct directory first."
    )
sys.path.insert(0, str(PROJECT_ROOT))

# Import modules
from configs.config import Config, set_global_seed
from data.dataset import load_retinamnist, make_mlp_loader
from representations.pca_repr import fit_transform_pca
from representations.ae_repr import train_and_extract_ae_representations
from representations.mae_repr import extract_mae_representations
from models.classifier import train_mlp
from utils.timer import Timer, log_time
from utils.metrics import compute_metrics, print_metrics_table
from utils.visualization import (
    plot_training_curves,
    plot_epoch_times,
    plot_reconstruction_samples,
    plot_tsne,
    plot_final_comparison,
)

os.makedirs(PROJECT_ROOT / "outputs", exist_ok=True)
print("Project root:", PROJECT_ROOT)

## SECTION 1 — Config & Dry Run Toggle

In [ ]:
# DRY RUN toggle
DRY_RUN = True   # ← CHANGE THIS TO FALSE FOR FULL TRAINING

config = Config(DRY_RUN=DRY_RUN)
set_global_seed(config.SEED)
print("Config:", config)

if config.DRY_RUN:
    print("🔍 DRY RUN MODE — using subsampled data")
else:
    print("🚀 FULL RUN MODE")

## SECTION 2 — Data Loading

In [ ]:
bundle = load_retinamnist(config, dry_run=config.DRY_RUN, num_workers=2)

(x_train, y_train) = bundle.arrays["train"]
(x_val, y_val) = bundle.arrays["val"]
(x_test, y_test) = bundle.arrays["test"]

print("n_train:", len(x_train), "n_val:", len(x_val), "n_test:", len(x_test))

def _class_dist(y):
    y = np.asarray(y).reshape(-1)
    vals, counts = np.unique(y, return_counts=True)
    return dict(zip(vals.tolist(), counts.tolist()))

print("Class dist (train):", _class_dist(y_train))
print("Class dist (val):", _class_dist(y_val))
print("Class dist (test):", _class_dist(y_test))

# Show a 4x4 grid of samples
fig, axes = plt.subplots(4, 4, figsize=(6, 6))
idxs = np.random.choice(len(x_train), size=16, replace=False)
for ax, idx in zip(axes.flatten(), idxs):
    ax.imshow(x_train[idx, 0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"y={int(y_train[idx])}")
    ax.axis("off")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "samples_grid.png", dpi=200, bbox_inches="tight")
plt.show()

## SECTION 3 — PCA Representations

In [ ]:
with Timer() as t_fit:
    pca_features, pca_model, pca_fit_time, pca_transform_times = fit_transform_pca(
        x_train, x_val, x_test, config
    )

log_time("PCA fit", pca_fit_time)
print("PCA transform times:", pca_transform_times)

# Explained variance (cumulative)
ev = np.cumsum(pca_model.explained_variance_ratio_)
plt.figure(figsize=(6, 4))
plt.plot(ev)
plt.xlabel("# Components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA Explained Variance (Cumulative)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "pca_explained_variance.png", dpi=200, bbox_inches="tight")
plt.show()

print("PCA feature shapes:", {k: v.shape for k, v in pca_features.items()})

## SECTION 4 — Autoencoder Training

In [ ]:
ae_train_loader = bundle.loaders["train"]
ae_val_loader = bundle.loaders["val"]

ae_start = time.time()
ae_features, ae_history, ae_model = train_and_extract_ae_representations(
    ae_train_loader,
    ae_val_loader,
    x_train,
    x_val,
    x_test,
    config,
)
ae_train_time = time.time() - ae_start

print("AE feature shapes:", {k: v.shape for k, v in ae_features.items()})
log_time("AE total train+extract", ae_train_time)

# Plot AE training loss curve
plt.figure(figsize=(6, 4))
plt.plot(ae_history["epoch_losses"], label="Train recon loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.title("AE Training Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "outputs" / "ae_loss_curve.png", dpi=200, bbox_inches="tight")
plt.show()

# Plot per-epoch times
plot_epoch_times(
    ae_history["epoch_times"],
    title="AE Epoch Times",
    save_path=str(PROJECT_ROOT / "outputs" / "ae_epoch_times.png"),
)
plt.show()

# Reconstruction samples
plot_reconstruction_samples(
    ae_model,
    ae_train_loader,
    device=config.DEVICE,
    n=8,
    save_path=str(PROJECT_ROOT / "outputs" / "ae_recon_samples.png"),
)
plt.show()

## SECTION 5 — MAE Feature Extraction

In [ ]:
mae_start = time.time()
mae_features, mae_times = extract_mae_representations(x_train, x_val, x_test, config)
mae_total_time = time.time() - mae_start

print("MAE feature shapes:", {k: v.shape for k, v in mae_features.items()})
print("MAE extraction times:", mae_times)
log_time("MAE total load+extract", mae_total_time)

# MAE feature dim should be 768 for ViT-Base
print("MAE train feature dim:", mae_features["train"].shape[1])

## SECTION 6 — MLP Classifier Training (All Three)

In [ ]:
def train_one_method(method_name: str, feats: dict, y_tr, y_va, cfg: Config):
    print(f"\n=== Training MLP for {method_name} ===")
    start = time.time()
    model, hist = train_mlp(
        feats["train"],
        y_tr,
        feats["val"],
        y_va,
        cfg,
        input_dim=int(feats["train"].shape[1]),
        num_classes=5,
    )
    train_time = time.time() - start
    log_time(f"{method_name} MLP train", train_time)
    return model, hist, train_time

mlp_models = {}
mlp_histories = {}
mlp_train_times = {}

mlp_models["PCA"], mlp_histories["PCA"], mlp_train_times["PCA"] = train_one_method(
    "PCA", pca_features, y_train, y_val, config
)
mlp_models["AE"], mlp_histories["AE"], mlp_train_times["AE"] = train_one_method(
    "AE", ae_features, y_train, y_val, config
)
mlp_models["MAE"], mlp_histories["MAE"], mlp_train_times["MAE"] = train_one_method(
    "MAE", mae_features, y_train, y_val, config
)

# Plot training curves side by side (3 columns)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, method in zip(axes, ["PCA", "AE", "MAE"]):
    h = mlp_histories[method]
    epochs = np.arange(1, len(h["train_loss"]) + 1)
    ax.plot(epochs, h["train_acc"], label="Train Acc")
    ax.plot(epochs, h["val_acc"], label="Val Acc")
    ax.set_title(f"{method} — Accuracy")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy")
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "outputs" / "mlp_acc_curves_all.png", dpi=200, bbox_inches="tight")
plt.show()

# Epoch time bar charts for all three
for method in ["PCA", "AE", "MAE"]:
    plot_epoch_times(
        mlp_histories[method]["epoch_times"],
        title=f"{method} MLP Epoch Times",
        save_path=str(PROJECT_ROOT / "outputs" / f"mlp_epoch_times_{method.lower()}.png"),
    )
    plt.show()

# Full curve plots (loss+acc) using utility (saved to disk)
for method in ["PCA", "AE", "MAE"]:
    plot_training_curves(
        mlp_histories[method],
        title=f"MLP on {method}",
        save_path=str(PROJECT_ROOT / "outputs" / f"mlp_curves_{method.lower()}.png"),
    )
    plt.show()

## SECTION 7 — Evaluation on Test Set

In [ ]:
@torch.no_grad()
def predict_numpy(model: torch.nn.Module, x: np.ndarray, batch_size: int = 256) -> np.ndarray:
    model.eval()
    preds = []
    for i in range(0, len(x), batch_size):
        xb = torch.from_numpy(np.asarray(x[i:i+batch_size], dtype=np.float32)).to(config.DEVICE)
        logits = model(xb)
        yhat = torch.argmax(logits, dim=1).detach().cpu().numpy().astype(int)
        preds.append(yhat)
    return np.concatenate(preds, axis=0)

results = {}

# Feature extraction / representation times
pca_feature_time = float(pca_fit_time + pca_transform_times["train"] + pca_transform_times["val"] + pca_transform_times["test"])
ae_feature_time = float(ae_train_time)  # includes AE train + feature extraction as measured
mae_feature_time = float(mae_total_time)  # includes model load + extraction

feature_times = {"PCA": pca_feature_time, "AE": ae_feature_time, "MAE": mae_feature_time}

for method in ["PCA", "AE", "MAE"]:
    x_te = {"PCA": pca_features, "AE": ae_features, "MAE": mae_features}[method]["test"]
    y_pred = predict_numpy(mlp_models[method], x_te)
    m = compute_metrics(y_test, y_pred)
    m["train_time"] = float(mlp_train_times[method])
    m["feature_time"] = float(feature_times[method])
    results[method] = m

print_metrics_table(results)

# Per-class accuracy details
for method in results:
    print(f"\n{method} per-class acc:", results[method]["per_class_acc"])

## SECTION 8 — Visualizations

In [ ]:
# t-SNE on TEST features (for consistent comparison)
features_for_tsne = {
    "PCA": pca_features["test"],
    "AE": ae_features["test"],
    "MAE": mae_features["test"],
}
plot_tsne(
    features_for_tsne,
    labels=y_test,
    save_path=str(PROJECT_ROOT / "outputs" / "tsne_test_all.png"),
)
plt.show()

# Final comparison bar chart (accuracy + macro_f1)
plot_final_comparison(results, save_path=str(PROJECT_ROOT / "outputs" / "final_comparison.png"))
plt.show()

print("Saved outputs to:", PROJECT_ROOT / "outputs")

## SECTION 9 — Summary Cell

In [ ]:
def _best_by(metric: str) -> tuple[str, float]:
    best_m = max(results.keys(), key=lambda k: results[k][metric])
    return best_m, float(results[best_m][metric])

best_acc_m, best_acc_v = _best_by("accuracy")

fastest_train_m = min(mlp_train_times.keys(), key=lambda k: mlp_train_times[k])
fastest_train_v = float(mlp_train_times[fastest_train_m])

# Simple tradeoff heuristic: accuracy divided by (train_time + feature_time)
trade = {
    m: float(results[m]["accuracy"]) / max(1e-9, float(results[m]["train_time"]) + float(results[m]["feature_time"]))
    for m in results
}
trade_m = max(trade.keys(), key=lambda k: trade[k])

print(
    f"Best accuracy: {best_acc_m} ({best_acc_v*100:.2f}%) | "
    f"Fastest training: {fastest_train_m} ({fastest_train_v:.2f}s) | "
    f"Best accuracy/time tradeoff: {trade_m}"
)